# ARIA Airborne Exploratory Data Analysis (EDA) with PySpark
**Author:** Brett Allen (brett.allen@gdit.com)

**Environment:**
```
   SageMaker Image: SparkAnalytics 3.0
  SageMaker Kernel: Glue PySpark
SageMaker Instance: ml.t3.medium (2 vCPU + 4 GiB @ $0.0416/hr) - EBS storage only (for HDFS)
```

**NOTE:** Must have the specified `%profile` available on system. Install AWS CLI via `pip install awscli` and run `aws configure` to initialize CLI environment. Must have generated keys in IAM first.

In [8]:
%iam_role arn:aws:iam::867344433302:role/endurasoft-GlueJobServiceRole
%profile default
%etl
%number_of_workers 2
%worker_type G.2X
%glue_version 3.0
%additional_python_modules "plotly-express,datashader,geopandas,shapely,dask[dataframe],folium"

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.7 
Current iam_role is arn:aws:iam::867344433302:role/service-role/SageMaker-ExecutionRole-20241012T165481
iam_role has been set to arn:aws:iam::867344433302:role/endurasoft-GlueJobServiceRole.
Previous profile: None
Setting new profile to: default
Previous session type: etl
Setting new session type to ETL
Previous number of workers: None
Setting new number of workers to: 2
Previous worker type: None
Setting new worker type to: G.2X
Setting Glue version to: 3.0
Additional python modules to be included:
plotly-express
datashader
geopandas
shapely
dask[dataframe]
folium


## Imports

In [1]:
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.window import Window
from pyspark import StorageLevel
from pyspark.context import SparkContext
import pyspark.sql.types as T
import pyspark.sql.functions as F
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.stat import Correlation
from pyspark.mllib.linalg.distributed import RowMatrix
import pandas as pd
import geopandas as gpd
import re
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
import plotly.express as px
from colorcet import fire
import datashader.transfer_functions as tf
import datashader as ds
import boto3
import io

Trying to create a Glue session for the kernel.
Session Type: etl
Worker Type: G.2X
Number of Workers: 2
Session ID: f1130f0e-4146-4223-86f8-2b240cd35dd3
Applying the following default arguments:
--glue_kernel_version 1.0.7
--enable-glue-datacatalog true
--additional-python-modules plotly-express,datashader,geopandas,shapely,dask[dataframe],folium
Waiting for session f1130f0e-4146-4223-86f8-2b240cd35dd3 to get into ready status...
Session f1130f0e-4146-4223-86f8-2b240cd35dd3 has been created.



In [10]:
%status

Session ID: f1130f0e-4146-4223-86f8-2b240cd35dd3
Status: READY
Role: arn:aws:iam::867344433302:role/endurasoft-GlueJobServiceRole
CreatedOn: 2024-11-29 15:18:19.881000+00:00
GlueVersion: 3.0
Session Type: glueetl
Idle Timeout: 2880
Timeout: 2880
Tags: {'owner': 'AIDA4T4OBYCLOCNMZTZU3', 'sagemaker:user-profile-arn': 'arn:aws:sagemaker:us-east-1:867344433302:user-profile/d-eikyi6u8iy8g/ballen', 'sagemaker:domain-arn': 'arn:aws:sagemaker:us-east-1:867344433302:domain/d-eikyi6u8iy8g', 'sagemaker:space-arn': 'arn:aws:sagemaker:us-east-1:867344433302:space/d-eikyi6u8iy8g/Bretts-Private-JupyterLab-Space'}
Worker Type: G.2X
Number of Workers: 2
Region: us-east-1
Applying the following default arguments:
--glue_kernel_version 1.0.7
--enable-glue-datacatalog true
--additional-python-modules plotly-express,datashader,geopandas,shapely,dask[dataframe],folium
Arguments Passed: ['--glue_kernel_version: 1.0.7', '--enable-glue-datacatalog: true', '--additional-python-modules: plotly-express,datashader

## Configurations

In [2]:
# Initialize spark context and glue context to create glue job for analysis
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

In [3]:
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
spark.conf.set("spark.sql.parquet.enableVectorizedReader","false")

In [4]:
s3_client = boto3.client('s3')

## Load the Data

### Track Points
Loading OpenSky Network data.
* Flight Recorder Data: `s3://endurasoft-dev-risk-framework/opensky-network/track-points/`

In [56]:
# NOTE: Loading 2024-08-01 through 2024-09-30 for proof of concept purposes.
# track_points_df = spark.read.parquet("s3://endurasoft-dev-risk-framework/opensky-network/track-points/year=2024/month=[8-9]/day=*/hour=*/*.parquet")
track_points_df = spark.read.parquet("s3://endurasoft-dev-risk-framework/opensky-network/track-points/year=2024/month=*/day=*/hour=*/*.parquet")
track_points_df.printSchema()

root
 |-- time: timestamp (nullable = true)
 |-- icao24: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- velocity: double (nullable = true)
 |-- heading: double (nullable = true)
 |-- vertrate: double (nullable = true)
 |-- callsign: string (nullable = true)
 |-- onground: boolean (nullable = true)
 |-- alert: boolean (nullable = true)
 |-- spi: boolean (nullable = true)
 |-- squawk: string (nullable = true)
 |-- baroaltitude: double (nullable = true)
 |-- geoaltitude: double (nullable = true)
 |-- lastposupdate: double (nullable = true)
 |-- lastcontact: double (nullable = true)
 |-- serials: array (nullable = true)
 |    |-- element: long (containsNull = true)


In [16]:
# NOTE: Full dataset is too large to persist; must target smaller subset to persist
# track_points_df.persist()

### UAS Sightings
Loading UAS sightings reports converted to tabular format.
* `s3://endurasoft-dev-risk-framework/datasets/uas_sightings_reports/zero_shot/uas_sightings.parquet`

In [6]:
uas_sightings_df = spark.read.parquet("s3://endurasoft-dev-risk-framework/datasets/uas_sightings_reports/zero_shot/uas_sightings.parquet")
uas_sightings_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- report_date: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- report_narrative: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- altitude: long (nullable = true)
 |-- uncertainty: string (nullable = true)


In [7]:
uas_sightings_df.persist()

DataFrame[id: bigint, report_date: string, city: string, state: string, report_narrative: string, timestamp: string, latitude: double, longitude: double, altitude: bigint, uncertainty: string]


### UAS Facility Maps (Grids)
Loading UAS Facility Map (UASFM) grids data. This data should be fused with UAS sightings data and track points data for risk analysis.
* `s3://endurasoft-dev-risk-framework/datasets/uasfm_grids/FAA_UAS_FacilityMap_Data.parquet`

In [8]:
uasfm_df = spark.read.parquet("s3://endurasoft-dev-risk-framework/datasets/uasfm_grids/FAA_UAS_FacilityMap_Data.parquet")
uasfm_df.printSchema()

root
 |-- OBJECTID: long (nullable = true)
 |-- CEILING: long (nullable = true)
 |-- UNIT: string (nullable = true)
 |-- MAP_EFF: string (nullable = true)
 |-- LAST_EDIT: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)
 |-- GLOBALID: string (nullable = true)
 |-- ARPT_COUNT: long (nullable = true)
 |-- APT1_FAAID: string (nullable = true)
 |-- APT1_ICAO: string (nullable = true)
 |-- APT1_NAME: string (nullable = true)
 |-- APT1_LAANC: long (nullable = true)
 |-- APT2_FAAID: string (nullable = true)
 |-- APT2_ICAO: string (nullable = true)
 |-- APT2_NAME: string (nullable = true)
 |-- APT2_LAANC: double (nullable = true)
 |-- APT3_FAAID: string (nullable = true)
 |-- APT3_ICAO: string (nullable = true)
 |-- APT3_NAME: string (nullable = true)
 |-- APT3_LAANC: double (nullable = true)
 |-- APT4_FAAID: double (nullable = true)
 |-- APT4_ICAO: double (nullable = true)
 |-- APT4_NAME: double (nullable = true)
 |-- APT4_LAANC: double

In [9]:
uasfm_df.persist()

DataFrame[OBJECTID: bigint, CEILING: bigint, UNIT: string, MAP_EFF: string, LAST_EDIT: string, LATITUDE: double, LONGITUDE: double, GLOBALID: string, ARPT_COUNT: bigint, APT1_FAAID: string, APT1_ICAO: string, APT1_NAME: string, APT1_LAANC: bigint, APT2_FAAID: string, APT2_ICAO: string, APT2_NAME: string, APT2_LAANC: double, APT3_FAAID: string, APT3_ICAO: string, APT3_NAME: string, APT3_LAANC: double, APT4_FAAID: double, APT4_ICAO: double, APT4_NAME: double, APT4_LAANC: double, APT5_FAAID: double, APT5_ICAO: double, APT5_NAME: double, APT5_LAANC: double, AIRS_COUNT: bigint, AIRSPACE_1: string, AIRSPACE_2: string, AIRSPACE_3: string, AIRSPACE_4: double, AIRSPACE_5: double, REGION: string, APT1_Enabled: string, APT2_Enabled: string, APT3_Enabled: string, APT4_Enabled: double, APT5_Enabled: double, Shape__Area: double, Shape__Length: double]


## Exploratory Data Analysis (EDA)

In [10]:
def clean_column_names(df):
    # Convert column names to lowercase and replace spaces, dots, and forward slashes with underscores
    return df.toDF(*[ re.sub(r'[ \.\/]+', '_', c.lower()) for c in df.columns ])

In [11]:
# Create function to show more information on missing values
def get_missing_values(df: DataFrame) -> DataFrame:
    """
    Create a dataframe to represent the missing values in the original dataframe.

    Args:
        df (DataFrame): Original dataframe to identify missing values.

    Returns:
        DataFrame: New dataframe representing a report of missing values in original dataframe.
    """
    total_records = df.count()
    
    # For each column, calculate the total missing, available, ratio, and percentage of missing values
    missing_stats = []
    
    for column in df.columns:
        total_missing = df.select(F.count(F.when(F.col(column).isNull(), column)).alias("total_missing")).collect()[0][0]
        total_available = total_records - total_missing
        ratio_missing = total_missing / total_records
        percent_missing = round(ratio_missing * 100, 1)
        
        missing_stats.append({
            'column': column,
            'total_missing': total_missing,
            'total_available': total_available,
            'ratio_missing': round(ratio_missing, 4),
            'percent_missing': f'{percent_missing}%'
        })
    
    # Create a DataFrame from the list of dictionaries
    result_df = df.sql_ctx.createDataFrame(missing_stats)
    
    # Ensure correct column order
    return result_df.select(['column', 'total_missing', 'total_available', 'ratio_missing', 'percent_missing'])

In [12]:
def get_unique_values(df):
    # NOTE: Can use approx_count_distinct (see https://stackoverflow.com/a/53764762)
    return df.agg(*(F.countDistinct(F.col(c)).alias(c) for c in df.columns))

### UAS Sightings

In [ ]:
# Clean column names
uas_sightings_df = clean_column_names(uas_sightings_df)

In [ ]:
# Convert boolean columns to integer
bool_cols = [col_name for col_name, col_type in uas_sightings_df.dtypes if col_type == "boolean"]

# Convert boolean columns to integers
for col_name in bool_cols:
    uas_sightings_df = uas_sightings_df.withColumn(col_name, F.col(col_name).cast(T.IntegerType()))

In [ ]:
uas_sightings_df.persist()

DataFrame[id: bigint, report_date: string, city: string, state: string, report_narrative: string, timestamp: string, latitude: double, longitude: double, altitude: bigint, uncertainty: string]


In [ ]:
uas_sightings_df.show(n=1, vertical=True, truncate=False)

-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 id               | 2293                                                                                                                                                                                                                                                                                                                                                                                                                                              
 report_date      | 2023-06-01                                                            

#### Descriptive Statistics

In [ ]:
uas_sightings_df.describe().show(n=len(uas_sightings_df.columns), vertical=True, truncate=False)

-RECORD 0------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 summary          | count                                                                                                                                                                                                                                                                                                                                                                                                                                            
 id               | 2274                                                                    

#### Date Range and Total Records

In [29]:
# Analyze date range for dataset
uas_sightings_df = uas_sightings_df.withColumn('report_date', F.to_timestamp('report_date'))

# Get the start and end dates
uas_sightings_start_date = uas_sightings_df.select(F.min("report_date")).collect()[0][0]
uas_sightings_end_date = uas_sightings_df.select(F.max("report_date")).collect()[0][0]

print("Start Date :", uas_sightings_start_date)
print("End Date   :", uas_sightings_end_date)

Start Date : 2023-06-01 00:00:00
End Date   : 2024-09-30 00:00:00


In [30]:
uas_sightings_date_diff = uas_sightings_end_date - uas_sightings_start_date
print(f'UAS Sightings dataset spans {uas_sightings_date_diff.days} days ({round(uas_sightings_date_diff.days/365, 2)} years)')

UAS Sightings dataset spans 487 days (1.33 years)


In [31]:
print(f'Total UAS sightings records: {uas_sightings_df.count():,}')

Total UAS sightings records: 2,274


#### Analyze Nulls

In [32]:
uas_sightings_missing_values_report = get_missing_values(uas_sightings_df)
uas_sightings_missing_values_report.show(n=len(uas_sightings_df.columns), truncate=False)

+----------------+-------------+---------------+-------------+---------------+
|column          |total_missing|total_available|ratio_missing|percent_missing|
+----------------+-------------+---------------+-------------+---------------+
|id              |0            |2274           |0.0          |0.0%           |
|report_date     |0            |2274           |0.0          |0.0%           |
|city            |0            |2274           |0.0          |0.0%           |
|state           |0            |2274           |0.0          |0.0%           |
|report_narrative|0            |2274           |0.0          |0.0%           |
|timestamp       |0            |2274           |0.0          |0.0%           |
|latitude        |0            |2274           |0.0          |0.0%           |
|longitude       |0            |2274           |0.0          |0.0%           |
|altitude        |0            |2274           |0.0          |0.0%           |
|uncertainty     |0            |2274           |0.0 

#### Analyze Uniqueness

In [33]:
uas_sightings_unique_counts = get_unique_values(uas_sightings_df)
uas_sightings_unique_counts.show(n=len(uas_sightings_df.columns), vertical=True, truncate=False)

-RECORD 0----------------
 id               | 2274 
 report_date      | 467  
 city             | 430  
 state            | 56   
 report_narrative | 2268 
 timestamp        | 729  
 latitude         | 807  
 longitude        | 976  
 altitude         | 168  
 uncertainty      | 2


In [13]:
# uas_sightings_unique_counts.toPandas()

**(TODO) Observation:**

#### Analyze Sightings by Altitude Bins

In [ ]:
alt_bins = [0, 150, 200, 250, 500, 750, 1000, 2500, 5000, 10000, float("inf")]
alt_labels = ['<150', '150-200', '200-250', '250-500', '500-750', '750-1000', '1000-2500', '2500-5000', '5000-10000', '>10000']

In [ ]:
# Create the altitude bins for altitude and aircraft_1_altitudeinfeet
uas_sightings_df = uas_sightings_df.withColumn("altitude_bin", 
                   F.when(F.col("altitude") < alt_bins[1], alt_labels[0])
                   .when((F.col("altitude") >= alt_bins[1]) & (F.col("altitude") < alt_bins[2]), alt_labels[1])
                   .when((F.col("altitude") >= alt_bins[2]) & (F.col("altitude") < alt_bins[3]), alt_labels[2])
                   .when((F.col("altitude") >= alt_bins[3]) & (F.col("altitude") < alt_bins[4]), alt_labels[3])
                   .when((F.col("altitude") >= alt_bins[4]) & (F.col("altitude") < alt_bins[5]), alt_labels[4])
                   .when((F.col("altitude") >= alt_bins[5]) & (F.col("altitude") < alt_bins[6]), alt_labels[5])
                   .when((F.col("altitude") >= alt_bins[6]) & (F.col("altitude") < alt_bins[7]), alt_labels[6])
                   .when((F.col("altitude") >= alt_bins[7]) & (F.col("altitude") < alt_bins[8]), alt_labels[7])
                   .when((F.col("altitude") >= alt_bins[8]) & (F.col("altitude") < alt_bins[9]), alt_labels[8])
                   .otherwise(alt_labels[9]))

In [ ]:
# Analyze event score by aircraft_0_altitude_bin
print('='*100)
print(f'UAS Sightings by Altitude Bins')
print('='*100)
uas_sightings_df.groupBy('altitude_bin').agg(
    F.count('*').alias('sightings_count'),
    F.min('*').alias('min_sightings'),
    F.max('*').alias('max_sightings'),
    F.avg('*').alias('average_sightings'),
    F.stddev('*').alias('stddev_sightings'),
).orderBy(F.col('average_sightings')).show()

### UASFM Grids

In [ ]:
uasfm_df = clean_column_names(uasfm_df)

In [ ]:
# Convert boolean columns to integer
bool_cols = [col_name for col_name, col_type in uasfm_df.dtypes if col_type == "boolean"]

# Convert boolean columns to integers
for col_name in bool_cols:
    uasfm_df = uasfm_df.withColumn(col_name, F.col(col_name).cast(T.IntegerType()))

In [ ]:
uasfm_df.show(n=1, vertical=True, truncate=False)

### Track Points

In [57]:
# Clean column names
track_points_df = clean_column_names(track_points_df)

In [58]:
# Convert boolean columns to integer
bool_cols = [col_name for col_name, col_type in track_points_df.dtypes if col_type == "boolean"]

# Convert boolean columns to integers
for col_name in bool_cols:
    track_points_df = track_points_df.withColumn(col_name, F.col(col_name).cast(T.IntegerType()))

In [17]:
# NOTE: Full dataset is too large to persist; must target smaller subset to persist
# track_points_df.persist()

In [54]:
track_points_df.show(n=1, vertical=True, truncate=False)

-RECORD 0----------------------------
 time          | 2024-08-03 04:50:21 
 icao24        | acffb7              
 lat           | null                
 lon           | null                
 velocity      | 92.65135011782233   
 heading       | 181.90915243299636  
 vertrate      | 13.980160000000001  
 callsign      | null                
 onground      | 0                   
 alert         | 0                   
 spi           | 0                   
 squawk        | null                
 baroaltitude  | null                
 geoaltitude   | null                
 lastposupdate | null                
 lastcontact   | 1.7226606201E9      
 serials       | [-1408233633]       
only showing top 1 row


#### Sync with UAS Sightings Date Range
Sync track points with UAS sightings date range for analysis.

In [34]:
synced_track_points_df = track_points_df.filter(F.col('time').between(uas_sightings_start_date, uas_sightings_end_date))

In [35]:
synced_track_points_count = synced_track_points_df.count()

In [40]:
print(f'Total track points after sync with UAS sightings: {synced_track_points_count:,}')

Total track points after sync with UAS sightings: 1,342,986,693


#### Save Synced Track Points
Saving smaller subset of track points data representing overlap with UAS sightings

In [37]:
synced_track_points_df = synced_track_points_df.withColumn('year', F.year('time'))\
                                               .withColumn('month', F.month('time'))\
                                               .withColumn('day', F.dayofmonth('time'))\
                                               .withColumn('hour', F.hour('time'))

In [38]:
# Write the DataFrame to Parquet format, partitioned by year, month, day, and hour
synced_track_points_df.write.partitionBy("year", "month", "day", "hour")\
                            .parquet("s3://endurasoft-dev-risk-framework/datasets/track-points/")

#### Load Synced Track Points

In [59]:
track_points_df = spark.read.parquet("s3://endurasoft-dev-risk-framework/datasets/track-points/year=*/month=*/day=*/hour=*/*.parquet")
track_points_df.printSchema()

root
 |-- time: timestamp (nullable = true)
 |-- icao24: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- velocity: double (nullable = true)
 |-- heading: double (nullable = true)
 |-- vertrate: double (nullable = true)
 |-- callsign: string (nullable = true)
 |-- onground: integer (nullable = true)
 |-- alert: integer (nullable = true)
 |-- spi: integer (nullable = true)
 |-- squawk: string (nullable = true)
 |-- baroaltitude: double (nullable = true)
 |-- geoaltitude: double (nullable = true)
 |-- lastposupdate: double (nullable = true)
 |-- lastcontact: double (nullable = true)
 |-- serials: array (nullable = true)
 |    |-- element: long (containsNull = true)


In [60]:
# Analyze date range for dataset
track_points_df = track_points_df.withColumn('time', F.to_timestamp('time'))

# Get the start and end dates
track_points_start_date = track_points_df.select(F.min("time")).collect()[0][0]
track_points_end_date = track_points_df.select(F.max("time")).collect()[0][0]

print("Start Date :", track_points_start_date)
print("End Date   :", track_points_end_date)

Start Date : 2023-10-01 00:00:01
End Date   : 2024-09-29 23:59:59


In [61]:
track_points_date_diff = track_points_end_date - track_points_start_date
print(f'OpenSky Network (track points synced with UAS sightings) dataset spans {track_points_date_diff.days} days ({round(track_points_date_diff.days/365, 2)} years)')

OpenSky Network (track points synced with UAS sightings) dataset spans 364 days (1.0 years)


In [62]:
track_points_count = track_points_df.count()

In [63]:
print(f'Total track points: {track_points_count:,}')

Total track points: 1,342,986,693


#### Analyze Number of Flights per Day

In [64]:
# Create day of week string column (e.g., "Monday", "Tuesday", ...)
track_points_df = track_points_df.withColumn('day_of_week', F.date_format('time', 'EEEE'))

In [65]:
# Create day of week number column (e.g., 1 = Sunday, 2 = Monday, etc.)
track_points_df = track_points_df.withColumn('daynum_of_week', F.dayofweek('time'))

In [66]:
print('='*100)
print(f'OpenSky Network Track Points by Day of Week')
print('='*100)
track_points_df.groupBy('day_of_week').agg(
    F.count('*').alias('track_points_count')
).orderBy(F.col('track_points_count')).show()

OpenSky Network Track Points by Day of Week
+-----------+------------------+
|day_of_week|track_points_count|
+-----------+------------------+
|    Tuesday|         184506992|
|     Monday|         189207528|
|   Saturday|         190917219|
|  Wednesday|         192431559|
|   Thursday|         193607343|
|     Friday|         195296543|
|     Sunday|         197019509|
+-----------+------------------+


In [ ]:
# Sample down to maximum of 1000 flights per day for each day spanning the date range of track points
# TODO Apply to aggregated results and then join with main track points dataframe on icao24 id to downsample
flights_per_day = 1000
day_window = Window.partitionBy("time", "daynum_of_week").orderBy(F.rand())
track_points_df = track_points_df.withColumn("row_num", F.row_number().over(day_window))
sampled_track_points_df = track_points_df.filter(F.col("row_num") <= flights_per_day)

In [ ]:
# Sample down to maximum of 1000 flights per day for each day spanning the date range of track points


#### Descriptive Statistics

In [55]:
track_points_df.describe().show(n=len(track_points_df.columns), vertical=True, truncate=False)

Py4JJavaError: An error occurred while calling o922.describe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 68 in stage 186.0 failed 4 times, most recent failure: Lost task 68.3 in stage 186.0 (TID 47491) (172.38.135.28 executor 1): java.io.IOException: No space left on device
	at sun.nio.ch.FileDispatcherImpl.write0(Native Method)
	at sun.nio.ch.FileDispatcherImpl.write(FileDispatcherImpl.java:60)
	at sun.nio.ch.IOUtil.writeFromNativeBuffer(IOUtil.java:93)
	at sun.nio.ch.IOUtil.write(IOUtil.java:65)
	at sun.nio.ch.FileChannelImpl.write(FileChannelImpl.java:211)
	at org.apache.spark.storage.CountingWritableChannel.write(DiskStore.scala:339)
	at java.nio.channels.Channels.writeFullyImpl(Channels.java:78)
	at java.nio.channels.Channels.writeFully(Channels.java:101)
	at java.nio.channels.Channels.access$000(Channels.java:61)
	at java.nio.channels.Channels$1.write(Channels.java:174)
	at java.io.BufferedOutputStream.flushBuffer(BufferedOutputStream.java:82)
	at j

#### Date Range and Total Records

In [ ]:
# Analyze date range for dataset
track_points_df = track_points_df.withColumn('time', F.to_timestamp('time'))

# Get the start and end dates
track_points_start_date = track_points_df.select(F.min("time")).collect()[0][0]
track_points_end_date = track_points_df.select(F.max("time")).collect()[0][0]

print("Start Date :", track_points_start_date)
print("End Date   :", track_points_end_date)

In [ ]:
track_points_date_diff = track_points_end_date - track_points_start_date
print(f'OpenSky Network (track points) dataset spans {track_points_date_diff.days} days ({round(track_points_date_diff.days/365, 2)} years)')

In [ ]:
track_points_count = track_points_df.count()

In [ ]:
print(f'Total track points: {track_points_count:,}')

#### Analyze Nulls

In [ ]:
track_points_missing_values_report = get_missing_values(track_points_df)
track_points_missing_values_report.show(n=len(track_points_df.columns), truncate=False)

Execution Interrupted. Attempting to cancel the statement (statement_id=44)


#### Analyze Uniqueness

In [ ]:
track_points_unique_counts = get_unique_values(track_points_df)
track_points_unique_counts.show(n=len(track_points_df.columns), vertical=True, truncate=False)

In [ ]:
# track_points_unique_counts.toPandas()

**(TODO) Observation:**

Low variance columns (categorical):
* List columns here

#### Encode Categorical Features
**NOTE**: Need to encode categorical string features with low variance and target only numeric features prior to identifying important/primary features

In [ ]:
track_points_categorical_features = [
    # TODO Create list of categorical features based on observations
]

encoded_features = [col + "_encoded" for col in track_points_categorical_features]

# Create a list of StringIndexer transformers
indexers = [StringIndexer(inputCol=col, outputCol=col + "_encoded") for col in track_points_categorical_features]

# Create a VectorAssembler to combine the indexed columns
assembler = VectorAssembler(inputCols=encoded_features, outputCol="features")

# Create a Pipeline to chain the transformers and assembler
pipeline = Pipeline(stages=indexers + [assembler])

# Fit and transform the DataFrame
indexer_model = pipeline.fit(track_points_df)
track_points_df_encoded = indexer_model.transform(track_points_df)

In [ ]:
track_points_df_encoded.select(*encoded_features).show(n=1, vertical=True, truncate=False)

In [ ]:
# Apply encoded changes
track_points_df = track_points_df_encoded

In [ ]:
# Drop features column that was used for indexer model
track_points_df = track_points_df.drop('features')

In [ ]:
track_points_df.show(n=1, vertical=True, truncate=False)

#### Identify Important/Primary Features

In [ ]:
track_points_df.dtypes

In [ ]:
numeric_cols = [c for c, t in track_points_df.dtypes if t in ('double', 'int', 'bigint', 'long', 'float')]
numeric_cols

In [ ]:
features = numeric_cols
target = 'eventscore'

# Create a VectorAssembler to combine features
assembler = VectorAssembler(inputCols=features, outputCol="rf_features", handleInvalid='skip')

# Create a RandomForestRegressor
rf = RandomForestRegressor(labelCol=target, featuresCol="rf_features", numTrees=100, maxBins=2000)

# Create a pipeline
pipeline = Pipeline(stages=[assembler, rf])

In [ ]:
# Split the data into training and testing sets
train_data, test_data = track_points_df.randomSplit([0.8, 0.2], seed=42)

In [ ]:
# Fit the random forest regressor model via pipeline (data cannot have any null values)
# TODO Need to analyze null data further and determine best approach for handling nulls (e.g., imputing null values or dropping rows with null values)
# NOTE: Setting handleInvalid='skip' for VectorAssembler will work around the null values issue
rf_model = pipeline.fit(train_data)

In [ ]:
# Make predictions on the test data
# predictions = rf_model.transform(test_data)

In [ ]:
# Evaluate the model
# evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="rmse")
# rmse = evaluator.evaluate(predictions)
# print("Root Mean Squared Error (RMSE):", rmse)

In [ ]:
# Analyze feature importances
feature_importances = rf_model.stages[1].featureImportances
feature_importances

In [ ]:
# Map feature importances back to column names and build pandas dataframe, sorted by importance in descending order
feature_importance_dict = dict(zip(assembler.getInputCols(), feature_importances))
feature_importance_df = pd.DataFrame({'Feature': feature_importance_dict.keys(), 'Importance': feature_importance_dict.values()})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

In [ ]:
pd.set_option('display.max_rows', 100)
feature_importance_df.round(6)

In [ ]:
f, ax = plt.subplots(figsize=(16, 8))
ax = sns.barplot(data=feature_importance_df, x='Importance', y='Feature')
ax.set_title("Visualize feature scores of the features", weight='bold')
ax.set_xlabel("Feature importance score")
ax.set_ylabel("Features")
plt.tight_layout()
plt.show()
%matplot plt

#### Analyze Correlations for Primary Features

In [ ]:
# Assemble features into a vector column
numeric_track_points_df = track_points_df.select(*numeric_cols)
assembler = VectorAssembler(inputCols=numeric_track_points_df.columns, outputCol="numeric_features", handleInvalid='skip')
numeric_track_points_df = assembler.transform(numeric_track_points_df)

# Calculate correlation matrix
corr_matrix = Correlation.corr(numeric_track_points_df, "numeric_features").head()[0].toArray()

In [ ]:
# Convert to pandas dataframe for plotting
corr_df = pd.DataFrame(corr_matrix, columns=numeric_track_points_df.columns[:-1], index=numeric_track_points_df.columns[:-1])

In [ ]:
corr_df

In [ ]:
# Plot heatmap
# See https://repost.aws/questions/QULOKRL66TSWy-WY4F9kL25w/how-to-get-glue-interactive-session-notebook-to-show-matplotlib-plot
plt.clf() # Clear current figure
fig = plt.figure(figsize=(32, 16))
sns.heatmap(corr_df, annot=True, fmt='.1f', annot_kws={"fontsize": 5}, cmap="coolwarm")
plt.title('Northern California TRACON Correlation Matrix')
plt.tight_layout()
# plt.savefig('./track_points_numeric_features_correlation_matrix.png') # NOTE: savefig does not work in glue pyspark. Need to shift+right click on the image and save manually
plt.show()
%matplot plt

In [ ]:
corr_df.to_csv('s3://endurasoft-dev-risk-framework/analysis/uas_risk_analysis_eda/track_points_numeric_features_correlation_matrix.csv', index=False)

In [ ]:
# Reduce number of features and select only features of interest
features_of_interest = [
    # TODO Create list representing features of interest
]

corr_df_foi = corr_df.loc[features_of_interest, :][features_of_interest]

# Plot heatmap
# See https://repost.aws/questions/QULOKRL66TSWy-WY4F9kL25w/how-to-get-glue-interactive-session-notebook-to-show-matplotlib-plot
plt.clf() # Clear current figure
fig = plt.figure(figsize=(30, 12))
sns.heatmap(corr_df_foi, annot=True, fmt='.1f', annot_kws={"fontsize": 12}, cmap="coolwarm")
plt.xticks(rotation=70)
plt.title('OpenSky Network Track Points Correlation Matrix (Features of Interest)', weight='bold')
plt.tight_layout()
plt.show()
%matplot plt

#### Analyze Covariance
Covariance measures the influence of change between features.

In [ ]:
numeric_track_points_df.columns

In [ ]:
vector_col = "cov_features"
assembler = VectorAssembler(inputCols=numeric_track_points_df.columns[:-1], outputCol=vector_col, handleInvalid="skip")
df_vector = assembler.transform(numeric_track_points_df).select(vector_col)

In [ ]:
# Must have single vector dtype
df_vector.dtypes

In [ ]:
row_matrix = RowMatrix(df_vector.rdd.map(list))
cov_matrix = row_matrix.computeCovariance()

In [ ]:
# Convert to pandas dataframe for plotting
cov_df = pd.DataFrame(cov_matrix.toArray(), columns=numeric_track_points_df.columns[:-1], index=numeric_track_points_df.columns[:-1])

In [ ]:
cov_df

In [ ]:
# Plot heatmap
# See https://repost.aws/questions/QULOKRL66TSWy-WY4F9kL25w/how-to-get-glue-interactive-session-notebook-to-show-matplotlib-plot
plt.clf() # Clear current figure
fig = plt.figure(figsize=(32, 16))
sns.heatmap(cov_df, annot=True, fmt='.1g', annot_kws={"fontsize": 5}, cmap="coolwarm")
plt.title('OpenSky Network Track Points Covariance Matrix')
plt.tight_layout()
plt.show()
%matplot plt

In [ ]:
cov_df_foi = cov_df.loc[features_of_interest, :][features_of_interest]

# Plot heatmap
# See https://repost.aws/questions/QULOKRL66TSWy-WY4F9kL25w/how-to-get-glue-interactive-session-notebook-to-show-matplotlib-plot
plt.clf() # Clear current figure
fig = plt.figure(figsize=(30, 12))
sns.heatmap(cov_df_foi, annot=True, fmt='.1g', annot_kws={"fontsize": 12}, cmap="coolwarm")
plt.xticks(rotation=70)
plt.title('OpenSky Network Track Points Covariance Matrix (Features of Interest)', weight='bold')
plt.tight_layout()
plt.show()
%matplot plt

* **Positive covariance**: The two variables tend to move in the same direction. 
* **Negative covariance**: The two variables tend to move in opposite directions. 
* **Zero covariance**: The two elements do not vary together.

#### Analyze Events by Altitude Bins

In [ ]:
alt_bins = [0, 150, 200, 250, 500, 750, 1000, 2500, 5000, 10000, float("inf")]
alt_labels = ['<150', '150-200', '200-250', '250-500', '500-750', '750-1000', '1000-2500', '2500-5000', '5000-10000', '>10000']

In [ ]:
# Create the altitude bins for baroaltitude
track_points_df = track_points_df.withColumn("altitude_bin", 
                   F.when(F.col("baroaltitude") < alt_bins[1], alt_labels[0])
                   .when((F.col("baroaltitude") >= alt_bins[1]) & (F.col("baroaltitude") < alt_bins[2]), alt_labels[1])
                   .when((F.col("baroaltitude") >= alt_bins[2]) & (F.col("baroaltitude") < alt_bins[3]), alt_labels[2])
                   .when((F.col("baroaltitude") >= alt_bins[3]) & (F.col("baroaltitude") < alt_bins[4]), alt_labels[3])
                   .when((F.col("baroaltitude") >= alt_bins[4]) & (F.col("baroaltitude") < alt_bins[5]), alt_labels[4])
                   .when((F.col("baroaltitude") >= alt_bins[5]) & (F.col("baroaltitude") < alt_bins[6]), alt_labels[5])
                   .when((F.col("baroaltitude") >= alt_bins[6]) & (F.col("baroaltitude") < alt_bins[7]), alt_labels[6])
                   .when((F.col("baroaltitude") >= alt_bins[7]) & (F.col("baroaltitude") < alt_bins[8]), alt_labels[7])
                   .when((F.col("baroaltitude") >= alt_bins[8]) & (F.col("baroaltitude") < alt_bins[9]), alt_labels[8])
                   .otherwise(alt_labels[9]))

In [ ]:
# Analyze event score by altitude_bin
print('='*100)
print(f'OpenSky Network Track Points Analysis by "altitude_bin"')
print('='*100)
track_points_df.groupBy('altitude_bin').agg(
    F.count('*').alias('track_points_count'),
    F.min('*').alias('min_track_points'),
    F.max('*').alias('max_track_points'),
    F.avg('*').alias('average_track_points'),
    F.stddev('*').alias('stddev_track_points'),
).orderBy(F.col('average_track_points')).show()

## Data Fusion
Use geospatial analysis to identify which UASFM grids the track points and uas sightings overlap and assign a unique grid ID accordingly to each record. The goal it to end up with a single cohesive dataset representing a join between track points, uas sightings, and UASFM grids, where each record has a UASFM global id associated with it.

In [ ]:
# TODO Create combined dataframe as join between track points, uas sightings, and uasfm grids
combined_df = None

### Aggregate Analysis
Analyze fused dataset for near mid-air collision (NMAC) and mid-air collision (MAC) risk for UAS.

In [ ]:
# Clean column names
combined_df = clean_column_names(combined_df)

In [ ]:
# Convert boolean columns to integer
bool_cols = [col_name for col_name, col_type in combined_df.dtypes if col_type == "boolean"]

# Convert boolean columns to integers
for col_name in bool_cols:
    combined_df = combined_df.withColumn(col_name, F.col(col_name).cast(T.IntegerType()))

In [ ]:
combined_df.persist()

#### Analyze Events by Facility

In [ ]:
combined_df.groupBy('facility').count().show()

In [ ]:
# Average event score by facility
combined_df.groupBy('facility').agg(
    F.count('eventscore').alias('event_count'),
    F.min('eventscore').alias('min_eventscore'),
    F.max('eventscore').alias('max_eventscore'),
    F.avg('eventscore').alias('average_eventscore'),
    F.stddev('eventscore').alias('stddev_eventscore'),
).show()

#### Analyze Events by Geolocation

##### Clustered Heatmap (Folium)

In [ ]:
cell_size = 0.007

df_grid = combined_df.withColumn("grid_lat", F.floor(combined_df["latitude"] / cell_size) * cell_size) \
             .withColumn("grid_lon", F.floor(combined_df["longitude"] / cell_size) * cell_size) \
             .groupBy("grid_lat", "grid_lon") \
             .agg(F.count("*").alias("point_count"))

In [ ]:
df_grid_pandas = df_grid.toPandas()

In [ ]:
df_grid_pandas

In [ ]:
map_center = [df_grid_pandas["grid_lat"].mean(), df_grid_pandas["grid_lon"].mean()]
m = folium.Map(location=map_center, zoom_start=10)

# Create the heatmap layer
HeatMap(
    data=df_grid_pandas[["grid_lat", "grid_lon", "point_count"]].values.tolist(),
    radius=15, 
    blur=10
).add_to(m)
m

In [ ]:
# s3://gdit-faa-datachallenge-proto/notebooks/aria_airborne_pyspark_eda/
map_filename = 'aria_airborne_events_geolocation.html'
buf = io.BytesIO(m._repr_html_().encode('utf-8'))
s3_client.put_object(Bucket='gdit-faa-datachallenge-proto', Key=f'notebooks/aria_airborne_pyspark_eda/{map_filename}', Body=buf)
buf.close()

##### Shader-based Point Density Heatmap (Plotly)

In [ ]:
# Apply moving average to reduce density of track points
# uniqueid
# latitude
# longitude
# ateventtime_timestamp - 2022-05-13T17:36:42.971Z         

In [ ]:
# ateventtime_timestamp - 2022-05-13T17:36:42.971Z -> format string: "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'" (can be inferred by pyspark)
combined_df = combined_df.withColumn('ateventtime_timestamp', F.to_timestamp('ateventtime_timestamp'))

In [ ]:
combined_df.select('ateventtime_timestamp').dtypes

In [ ]:
# Define the window specification
# window_spec = Window.partitionBy("uniqueid").orderBy("ateventtime_timestamp").rowsBetween(-1, 1)

# # Calculate the moving average for latitude and longitude
# windowed_df = combined_df.withColumn("avg_latitude", F.avg("latitude").over(window_spec))
# windowed_df = windowed_df.withColumn("avg_longitude", F.avg("longitude").over(window_spec))

# # Select the desired columns (including the moving averages)
# windowed_df = windowed_df.select("uniqueid", "ateventtime_timestamp", "avg_latitude", "avg_longitude")

# # Drop duplicates (optional, to further reduce density)
# mav_df = windowed_df.dropDuplicates(["uniqueid", "avg_latitude", "avg_longitude"])

# mav_df.show(n=1, vertical=True, truncate=False)

In [ ]:
# Determine average time diff in seconds between events; this will help determine reduction ratio
# Calculate the time difference between consecutive records
window = Window.orderBy(F.col("ateventtime_timestamp"))
combined_df = combined_df.withColumn("prev_timestamp", F.lag("ateventtime_timestamp").over(window))
combined_df = combined_df.withColumn("time_diff", F.unix_timestamp("ateventtime_timestamp") - F.unix_timestamp("prev_timestamp"))

# Calculate the average time difference in seconds
avg_time_diff = combined_df.agg(F.avg("time_diff")).collect()[0][0]

print("Average time difference in seconds:", avg_time_diff)

In [ ]:
# Create year, month, day, hour, minute, seconds columns
combined_df = combined_df.withColumn("year", F.year("ateventtime_timestamp")) \
                         .withColumn("month", F.month("ateventtime_timestamp")) \
                         .withColumn("day", F.dayofmonth("ateventtime_timestamp")) \
                         .withColumn("hour", F.hour("ateventtime_timestamp")) \
                         .withColumn("minute", F.minute("ateventtime_timestamp")) \
                         .withColumn("second", F.second("ateventtime_timestamp"))

In [ ]:
# Apply date range based windowing based on https://stackoverflow.com/a/45824339
# NOTE: This will take a while to run depending on the size of the dataset
# Function to calculate number of seconds from number of days
days_to_seconds = lambda i: i * 86400

# Create window by casting timestamp to long (number of seconds)
# https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Window.rangeBetween.html
# https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Window.rowsBetween.html
window_spec = Window.partitionBy("uniqueid").orderBy(F.col("ateventtime_timestamp").cast('long')).rangeBetween(-days_to_seconds(1), 0)
# window_spec = Window.orderBy(F.col('ateventtime_timestamp')).rowsBetween(-1, 0)
# window_spec = Window.partitionBy("facility").orderBy(F.col("ateventtime_timestamp")).rowsBetween(Window.currentRow, 1)

# Calculate the moving average for latitude and longitude
windowed_df = combined_df.withColumn("avg_latitude", F.avg("latitude").over(window_spec))
windowed_df = windowed_df.withColumn("avg_longitude", F.avg("longitude").over(window_spec))

# Calculate the moving average for event score
# windowed_df = combined_df.withColumn("avg_eventscore", F.avg("eventscore").over(window_spec))

# Drop duplicates to further reduce density
mav_df = windowed_df.dropDuplicates(["uniqueid", "avg_latitude", "avg_longitude"])
# mav_df = windowed_df.dropDuplicates(["avg_eventscore"])
# mav_df.show(n=1, vertical=True, truncate=False)

In [ ]:
# Analyze density reduction with moving average
# total starting records = 3,590,562
#   reduced to 3,039,839 with rowsBetween(-2, 0)
#   reduced to 2,897,022 with rowsBetween(-1, 0)
#   reduced to 2,396,445 with rowsBetween(Window.currentRow, 1)
f'{combined_df.count():,}', f'{mav_df.count():,}'

In [ ]:
reduced_df = combined_df.withColumn('trunc_timestamp', F.date_trunc('second', F.col('ateventtime_timestamp')))\
                        .groupBy('trunc_timestamp')\
                        .agg(*[F.first(col).alias(col) for col in combined_df.columns])\
                        .select(*combined_df.columns)\
                        .orderBy(F.col('ateventtime_timestamp').asc())

In [ ]:
# Analyze density reduction with date truncation
f'{combined_df.count():,}', f'{reduced_df.count():,}'

In [ ]:
sampled_df = combined_df.sample(fraction=0.1, seed=42)

In [ ]:
# Analyze density reduction with sampling
f'{combined_df.count():,}', f'{sampled_df.count():,}'

In [ ]:
sampled_df_pandas = sampled_df.toPandas()

In [ ]:
# Plot Latitude & Longitude on a map using plotly and datashader - https://plotly.com/python/datashader/
cvs = ds.Canvas(plot_width=1000, plot_height=1000)
agg = cvs.points(sampled_df_pandas, x='longitude', y='latitude')
coords_lat, coords_lon = agg.coords['latitude'].values, agg.coords['longitude'].values

# Corners of the image, which need to be passed to mapbox
coordinates = [[coords_lon[0], coords_lat[0]],
               [coords_lon[-1], coords_lat[0]],
               [coords_lon[-1], coords_lat[-1]],
               [coords_lon[0], coords_lat[-1]]]

img = tf.shade(agg, cmap=fire)[::-1].to_pil()

# Trick to create rapidly a figure with mapbox axes
fig = px.scatter_mapbox(
    sampled_df_pandas[:1], # First row, all columns
    lat='latitude', 
    lon='longitude',
    opacity=0.0, # Hidden
    zoom=10,
    height=600,
    width=1600,
    center={'lat': 36.778259, 'lon': -119.417931}, # California
)

# Add the datashader image as a mapbox layer image
fig.update_layout(
    mapbox_style="carto-darkmatter",
    mapbox_layers = [
        {
            "sourcetype": "image",
            "source": img,
            "coordinates": coordinates
        }
    ],
    margin=dict(l = 0, r = 0, t = 0, b = 0),
)
fig.show()

In [ ]:
# Save the plotly heatmap
string_buf = io.StringIO()
fig.write_html(string_buf)
string_buf.seek(0)

string_data = string_buf.getvalue()
bytes_buf = io.BytesIO(string_data.encode('utf-8'))

# s3://gdit-faa-datachallenge-proto/notebooks/aria_airborne_pyspark_eda/
fig_filename = 'aria_airborne_events_plotly_heatmap_shader.html'
s3_client.put_object(Bucket='gdit-faa-datachallenge-proto', Key=f'notebooks/aria_airborne_pyspark_eda/{fig_filename}', Body=bytes_buf)
bytes_buf.close()
string_buf.close()

#### KMeans Clustering by Location

In [ ]:
combined_df = combined_df.drop('location_features')

In [ ]:
# TODO determine the optimal number of clusters (k)
k = 10 
kmeans = KMeans(k=k, featuresCol="location_features")

# Assemble features into a vector
assembler = VectorAssembler(inputCols=["latitude", "longitude"], outputCol="location_features")
combined_df = assembler.transform(combined_df)

# Train the model
kmeans_model = kmeans.fit(combined_df)

# Assign cluster labels to each track point
combined_df = kmeans_model.transform(combined_df)

In [ ]:
combined_df.show(n=1, vertical=True, truncate=True)

In [ ]:
combined_df.groupBy('prediction').count().show()

## Clean up Resources

In [19]:
%stop_session

Stopping session: f1130f0e-4146-4223-86f8-2b240cd35dd3
Stopped session.
